In [ ]:
!pip install -q sympy numpy transformers accelerate tqdm antlr4-python3-runtime==4.11.1 vllm

import os
os.kill(os.getpid(), 9)

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
import json
import os

MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 8192

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from tqdm import tqdm

In [ ]:
import torch
print(torch.cuda.is_available())           # Must be True
print(torch.cuda.get_device_name(0))       # Should say "Tesla T4"
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")  # ~15 GB

In [ ]:
# Data: put the question file at data/public.jsonl (judger.py lives in the repo root)
assert os.path.exists("data/public.jsonl"), "Put the dataset at data/public.jsonl first"
assert os.path.exists("judger.py"), "Run this notebook from the repo root so judger.py is importable"
import shutil, os

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)


In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

In [ ]:
for i in data:
    if 'options' not in i and ('A.' in i["question"] or 'B.' in i["question"]):
        print(i)

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are a world-class mathematician."
    "Put your final answer inside \\boxed{}. "
    "Solve the problem. Then independently verify your answer by checking it from a different angle or plugging it back in."
    "Be very concise"
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
     "You are a world-class mathematician."
     "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
     "Be very concise"
)

OTHER = (
    "You are a world-class mathematician."
     "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
     "Be very concise"
     "you can pick multiple answers if they exist"
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    elif options is None and ('A.' in question or 'B.' in question):
      return OTHER, question
    return SYSTEM_PROMPT_MATH, question





for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)


In [ ]:
batch_size=4
prompts = []
# load results.Json
#built set of completed ids

for item in data:
    # skip over ids that are already completed
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
        #enable_thinking=False
    )
    prompts.append(prompt_text)

print(f"Generating responses for {len(prompts)} questions...")

responses = []
for i in range(0, len(prompts), batch_size):

  inputs = tokenizer(
      prompts[i:i+batch_size],
      return_tensors="pt",
      padding=True,
      truncation=True,
      max_length=2048,
  ).to(llm.device)

  torch.cuda.empty_cache()
  with torch.no_grad():
      output_ids = llm.generate(
          **inputs,
          max_new_tokens=MAX_TOKENS,
          temperature=0.6,
          top_p=0.95,
          top_k=20,
          repetition_penalty=1.0,
          do_sample=True,
          pad_token_id=tokenizer.eos_token_id,
      )

for i, out in enumerate(output_ids):
    new_tokens = out[inputs["input_ids"].shape[1]:]
    responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

In [ ]:
SAVE_EVAL = True

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")